# Yelp dataset build and prompt generation

This notebook builds an interaction dataset from the Yelp Open Dataset and generates prompt inputs.

**Pipeline:**
- Load `review.json` (JSONL) → optional time filtering (2019) → iterative 5-core filtering.
- Load `business.json` (JSONL) → flatten to a table → keep only businesses appearing in interactions.
- Reindex users/items to contiguous integer IDs and save `inter.csv`, `meta.csv`.
- Build item prompt inputs (`item_prompt_input.pkl`).


In [ ]:
import os
import json
import random
import pickle
import pandas as pd

In [ ]:
# Download the Yelp Open Dataset JSON files (business + review).
# https://business.yelp.com/data/resources/open-dataset/
DATASET = 'Yelp'
RAW_PATH = os.path.join('./data/', DATASET)

items = RAW_PATH + "/yelp_academic_dataset_business.json"
reviews = RAW_PATH + "/yelp_academic_dataset_review.json"

# Review data

In [ ]:
# Load Yelp reviews (JSON Lines: one JSON object per line)
# filter reviews by date (we only use records from 2019)
review_list = []

with open(reviews, 'r') as file:
    for line in file:
        line = line.strip()

        if line: 
            data_cur = json.loads(line)
            review_list.append(data_cur)

review_df = pd.DataFrame(review_list)
review_df['date'] = pd.to_datetime(review_df['date'])
review_df_filtered = review_df[(review_df['date'] >= '2019-01-01') & (review_df['date'] < '2020-01-01')]

In [ ]:
def filter_df_5core(df, min_interactions: int = 5):
    """
    Iteratively apply k-core filtering on (user_id, business_id).
    Removes businesses with < min_interactions reviews and users with < min_interactions reviews,
    repeating until the dataset is stable.
    """

    while True:
        print(df.shape)
        business_counts = df['business_id'].value_counts()
        user_counts = df['user_id'].value_counts()

        businesses_to_remove = business_counts[business_counts < min_interactions].index
        users_to_remove = user_counts[user_counts < min_interactions].index

        if len(businesses_to_remove) == 0 and len(users_to_remove) == 0:
            break

        df = df[~df['business_id'].isin(businesses_to_remove)]
        df = df[~df['user_id'].isin(users_to_remove)]

    return df


# Apply 5-core filtering after the optional time filter above
review_df_filtered_5core = filter_df_5core(review_df_filtered, min_interactions=5)

# Meta file

In [ ]:
# We only keep the fields used downstream (no need to flatten nested attributes/hours).
BUSINESS_FIELDS = ['business_id', 'name', 'categories', 'city', 'state']
business_records = []

with open(items, 'r') as fin:
    for line in fin:
        if not line.strip():
            continue

        obj = json.loads(line)
        business_records.append({k: obj.get(k) for k in BUSINESS_FIELDS})

meta_df = pd.DataFrame(business_records, columns=BUSINESS_FIELDS)
useful_meta_df = meta_df[meta_df['business_id'].isin(review_df_filtered_5core['business_id'])].reset_index(drop=True).copy()

In [ ]:
# missing-rate check for the kept metadata fields
missing_rate = (useful_meta_df.isnull().sum() / useful_meta_df.shape[0]).sort_values(ascending=False)
missing_rate.map(lambda x: "{:.2%}".format(x))

In [ ]:
# Handle missing values: unify all fields to "missing or unknown"
for col in ['name', 'categories', 'city', 'state']:
    useful_meta_df[col] = useful_meta_df[col].fillna('missing or unknown')

useful_meta_df

# Build dataset

In [ ]:
# Keep raw Yelp identifiers explicit to avoid confusion.
out_df = review_df_filtered_5core.copy().rename(columns={'date': 'time'})
out_df = out_df[['user_id', 'business_id', 'time']]
out_df = out_df.drop_duplicates(['user_id', 'business_id', 'time'])
out_df = out_df.sort_values(by=['user_id', 'time'], kind='mergesort').reset_index(drop=True)
out_df.head()

In [ ]:
uids = out_df['user_id'].unique()
user2id = dict(zip(uids, range(1, len(uids) + 1)))
bids = out_df['business_id'].unique()
item2id = dict(zip(bids, range(1, len(bids) + 1)))

with open(RAW_PATH+'/item2id.json', 'w') as f:
    json.dump(item2id, f, indent=4)

with open(RAW_PATH+'/user2id.json', 'w') as f:
    json.dump(user2id, f, indent=4)

# Add integer IDs and drop raw columns before saving downstream files
out_df['user_id'] = out_df['user_id'].apply(lambda x: user2id[x])
out_df['item_id'] = out_df['business_id'].apply(lambda x: item2id[x])
out_df = out_df.drop(columns=['business_id'])
out_df.head()

In [ ]:
# Attach integer item_id to metadata
useful_meta_df.loc[:, 'item_id'] = useful_meta_df['business_id'].apply(lambda x: item2id[x])

# save data
out_df.to_csv(RAW_PATH + '/inter.csv', index=False)
useful_meta_df.to_csv(RAW_PATH + '/meta.csv', index=False)

inter = out_df.drop(columns=['time'])
inter.columns = ['user_id:token', 'item_id:token']
inter.to_csv(RAW_PATH + '/' + DATASET +'.inter', sep='	', index=False)

df_txt = out_df.drop(columns=['time'])
df_txt.to_csv(RAW_PATH + '/' + DATASET+'.txt', sep=' ', index=False, header=False)

# Item prompt

In [ ]:
# Item Prompt: prepare item metadata and short user histories

# ----- metadata preparation -----
item_prompt_meta_df = pd.read_csv(RAW_PATH + '/meta.csv', low_memory=False)
for col in ['name', 'categories', 'city', 'state']:
    item_prompt_meta_df[col] = item_prompt_meta_df[col].fillna('missing or unknown')
meta_dict = item_prompt_meta_df.set_index('item_id').T.to_dict()

# ----- history sequence preparation -----
inter = pd.read_csv(RAW_PATH + '/inter.csv')
inter = inter.sort_values(by=['user_id', 'time'], kind='mergesort').reset_index(drop=True)
_pos = inter.groupby('user_id').cumcount()
_n = inter.groupby('user_id')['user_id'].transform('size')
inter = inter[_pos < (_n - 2)].reset_index(drop=True)


# seq slide augmentation
def prepare_data_augmentation(df):
    max_item_list_len = 20
    last_uid = None
    uid_list, item_list, target, item_list_length = [], [], [], []

    for _, row in df.iterrows():
        uid, item_id = row['user_id'], row['item_id']

        if last_uid != uid:
            last_uid = uid
            seq = []

        else:
            if len(seq) > max_item_list_len:
                seq = seq[1:]

            uid_list.append(uid)
            item_list.append(seq[:])
            target.append(item_id)
            item_list_length.append(len(seq))

        seq.append(item_id)

    return uid_list, item_list, target, item_list_length


uid_list, item_list, target, item_list_length = prepare_data_augmentation(inter)

In [ ]:
filtered_list = [sublist[-11:] for sublist in item_list if (len(sublist) >= 2 and len(sublist) <= 21)]
print("length of filtered_list", len(filtered_list))

last_values = set(sublist[-1] for sublist in filtered_list)
end_dict = {value: [] for value in last_values}

for sublist in filtered_list:
    last_value = sublist[-1]
    end_dict[last_value].append(sublist)

random.seed(2026)

for key in end_dict:
    if len(end_dict[key]) > 5:
        end_dict[key] = random.sample(end_dict[key], 5)

end_dict_text = {}
for key in end_dict:
    end_dict_text[key] = []

    for ls in end_dict[key]:
        ls_text = [meta_dict[item]['name'] for item in ls]
        ls_text = [val for val in ls_text if val != "missing or unknown"]
        end_dict_text[key].append(ls_text)

end_dict_text_formatted = {}
for item_id in end_dict_text.keys():
    same_target_seqs = ''

    for seq in end_dict_text[item_id]:
        seq = ' -> '.join(seq)
        seq = '[' + seq + '] # '
        seq += " \n "
        same_target_seqs += seq

    end_dict_text_formatted[item_id] = same_target_seqs

In [ ]:
def item_template(item_dict):
    # Same fields as meta.csv: name, categories, city, state (no stars).
    name = item_dict.get('name', 'missing or unknown')
    if pd.isna(name) or name == '' or name == 'nan':
        name = 'missing or unknown'

    categories = item_dict.get('categories', 'missing or unknown')
    if pd.isna(categories) or categories == '' or categories == 'nan':
        categories = 'missing or unknown'

    city = item_dict.get('city', 'missing or unknown')
    if pd.isna(city) or city == '' or city == 'nan':
        city = 'missing or unknown'

    state = item_dict.get('state', 'missing or unknown')
    if pd.isna(state) or state == '' or state == 'nan':
        state = 'missing or unknown'

    # Wording aligned with Yelp_backup.ipynb (without average rating / stars).
    text = (
        f"The business's name is {name}; "
        f"The business's categories are {categories}; "
        f"The business is located at {city}, {state}; "
    )

    return text


item_meta_formatted = {}
for item_id in item2id.values():
    item_meta_formatted[item_id] = item_template(meta_dict[item_id])


def complete_prompt_gen(item_info, history):
    prompt = f"""
    Assume you are an expert for local business recommendation. Please help me analyze a specific business. You are provided with the following information:
    1) The attributes of the business: {item_info}
    2) Historical interaction sequences ending at this business: {history}

    Input format notes for historical sequences:
    - Different sequences are separated by '#'.
    - Each sequence is in LIST format and items are separated by '->'.
    - The last item in each sequence is this target business.

    Requirements:
    1) Please briefly describe the given business.
    2) Please analyze what type of users would enjoy this business, using the business categories and location together with patterns in the historical sequences when available.

    Please provide your answer in JSON format, following this structure:
    {{
    "item summary": "A description of the business, no more than 80 words.",
    "potential user analysis": "What type of users would enjoy this business, no more than 50 words."
    }}
    """

    return prompt

item_prompt = {}
for item_id in item2id.values():
    item_info = item_meta_formatted[item_id]

    if item_id in end_dict_text_formatted.keys():
        history = end_dict_text_formatted[item_id]
    else:
        history = ' [None] '

    item_prompt[item_id] = complete_prompt_gen(item_info, history)

with open(RAW_PATH + '/item_prompt_input.pkl', 'wb') as pickle_file:
    pickle.dump(item_prompt, pickle_file)